# Step 5 - manual claim review

Run this notebook from the project folder; it uses the existing Step 3/4 files and does not rerun Step 4. The context-sheet cell writes a separate `annotation_sheet_with_context.csv` without changing the existing blind sheet or key. It includes the complete multi-sentence LLM explanation, the associated sentence, and the claim text. The model identity and its separate self-verdict line are hidden by default to reduce rating bias.

`sentence` is the source sentence Step 4 associated with a splitter quote; it is not guaranteed to contain all of `claim_text`. The new sheet includes flags showing whether the claim text appears verbatim in that sentence or anywhere in the complete explanation. If it does not, use the complete text and board as context; note the mismatch and use `unknown` if the intended claim is unclear.

Your ratings are saved separately to `outputs/step4_claims/<month>/manual_annotations_step5.csv`; the original blinded sheet and key are not modified. Do not rerun Step 4 after annotating, because Step 4 regenerates those files.

In [1]:
import html
import json
import re
from pathlib import Path

import chess
import chess.svg
import pandas as pd
from IPython.display import HTML, display

import config

PROJECT = Path.cwd()
if not (PROJECT / "config.py").is_file():
    raise RuntimeError("Open/run this notebook with the project folder as the working directory.")

STEP3_DIR = config.OUT_DIR / "step3_llm_pilot" / config.MONTH
STEP4_DIR = config.OUT_DIR / "step4_claims" / config.MONTH
SHEET_PATH = STEP4_DIR / "annotation_sheet_blind.csv"
KEY_PATH = STEP4_DIR / "annotation_key.csv"
MANUAL_PATH = STEP4_DIR / "manual_annotations_step5.csv"

for required_path in (SHEET_PATH, KEY_PATH, STEP3_DIR):
    if not required_path.exists():
        raise FileNotFoundError(f"Required Step 3/4 input is missing: {required_path}")

sheet = pd.read_csv(SHEET_PATH, dtype={"blind_id": str}, keep_default_na=False)
crosswalk = pd.read_csv(KEY_PATH, dtype={"blind_id": str}, keep_default_na=False)
required_sheet_columns = {"blind_id", "fen", "side_to_move", "move_to_explain_san", "sentence", "claim_text"}
required_key_columns = {"blind_id", "model", "condition", "repeat", "pilot_id"}
if not required_sheet_columns.issubset(sheet.columns):
    raise ValueError(f"Blind sheet is missing columns: {sorted(required_sheet_columns - set(sheet.columns))}. Rerun the updated step4_claims.ipynb before annotating.")
if not required_key_columns.issubset(crosswalk.columns):
    raise ValueError(f"Annotation key is missing columns: {sorted(required_key_columns - set(crosswalk.columns))}")
for optional_column in ("source_quote", "source_subject"):
    if optional_column not in sheet.columns:
        sheet[optional_column] = ""
if sheet["blind_id"].duplicated().any() or crosswalk["blind_id"].duplicated().any():
    raise ValueError("blind_id must be unique in both Step 4 annotation files.")
if set(sheet["blind_id"]) != set(crosswalk["blind_id"]):
    raise ValueError("The blind sheet and annotation key have different blind_id values.")

generations = {}
generation_files = sorted(STEP3_DIR.glob("*/generations.jsonl"))
if not generation_files:
    raise FileNotFoundError(f"No generations.jsonl files found under {STEP3_DIR}")
for generation_file in generation_files:
    with generation_file.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            generation_id = (str(record["model"]), str(record["condition"]), int(record["repeat"]), str(record["pilot_id"]))
            if generation_id in generations:
                raise ValueError(f"Duplicate generation record found in {generation_file}, line {line_number}.")
            generations[generation_id] = record

for row in crosswalk.to_dict(orient="records"):
    generation_id = (str(row["model"]), str(row["condition"]), int(row["repeat"]), str(row["pilot_id"]))
    if generation_id not in generations:
        raise KeyError(f"No matching Step 3 generation exists for blind row {row['blind_id']}.")
    record = generations[generation_id]
    if record.get("truncated") or record.get("think_unterminated") or not str(record.get("text", "")).strip():
        raise ValueError(f"The generation mapped to {row['blind_id']} is empty or unusable.")

sheet = sheet.set_index("blind_id", drop=False)
crosswalk = crosswalk.set_index("blind_id", drop=False)
print(f"Loaded {len(sheet)} blinded claims and matched their original generations.")

Loaded 161 blinded claims and matched their original generations.


In [2]:
VERDICT_RE = re.compile(r"(?im)^[ \t>*_#-]*verdict[ \t*_]*:[ \t*_]*(good|inaccurate|bad)\b[^\n]*$")
EXPLANATION_LABEL_RE = re.compile(r"(?im)^[ \t*_#-]*explanation[ \t*_]*:[ \t*_]*")

def explanation_without_self_verdict(text):
    matches = list(VERDICT_RE.finditer(text))
    if matches:
        match = matches[-1]
        text = text[:match.start()] + text[match.end():]
    return EXPLANATION_LABEL_RE.sub("", text).strip()

def normalize_for_match(text):
    return re.sub(r"\s+", " ", str(text)).strip().casefold()

def show_position(fen, move_san, side_to_move=None, size=360):
    board = chess.Board(str(fen))
    if side_to_move is not None:
        side = str(side_to_move).strip().lower()
        expected_turn = {"white": chess.WHITE, "black": chess.BLACK}.get(side)
        if expected_turn is None:
            raise ValueError(f"side_to_move must be 'white' or 'black', got {side_to_move!r}.")
        if board.turn != expected_turn:
            raise ValueError("The FEN side-to-move field does not match the side_to_move value.")

    move = board.parse_san(str(move_san).strip())
    orientation = board.turn
    before_svg = chess.svg.board(
        board,
        size=size,
        orientation=orientation,
        arrows=[chess.svg.Arrow(move.from_square, move.to_square, color="#1769aa")],
    )
    after = board.copy()
    after.push(move)
    after_svg = chess.svg.board(after, size=size, orientation=orientation, lastmove=move)
    move_label = html.escape(str(move_san))
    display(HTML(
        '<div style="display:flex;gap:24px;flex-wrap:wrap;align-items:flex-start">'
        f'<div><b>Before the move</b><br><span>Arrow shows {move_label}</span><br>{before_svg}</div>'
        f'<div><b>After {move_label}</b><br>{after_svg}</div>'
        '</div>'
    ))

def show_claim(blind_id, include_model_verdict=False):
    blind_id = str(blind_id).strip().upper()
    if blind_id not in sheet.index:
        raise KeyError(f"Unknown blind_id {blind_id!r}. Check annotation_sheet_blind.csv.")

    row = sheet.loc[blind_id]
    hidden_key = crosswalk.loc[blind_id]
    generation_id = (str(hidden_key["model"]), str(hidden_key["condition"]), int(hidden_key["repeat"]), str(hidden_key["pilot_id"]))
    generated = generations[generation_id]
    body = explanation_without_self_verdict(str(generated["text"]))
    shown_text = str(generated["text"]) if include_model_verdict else body
    text_label = (
        "Full raw model output (includes its self-verdict; use only after your independent rating)"
        if include_model_verdict
        else "LLM explanation (self-verdict line hidden during blind rating)"
    )

    display(HTML(f"<h2>Blind claim {html.escape(blind_id)}</h2>"))
    display(HTML(
        f"<p><b>Side to move:</b> {html.escape(str(row.side_to_move))}<br>"
        f"<b>Move being explained:</b> {html.escape(str(row.move_to_explain_san))}<br>"
        f"<b>FEN:</b> <code>{html.escape(str(row.fen))}</code></p>"
    ))
    show_position(row.fen, row.move_to_explain_san, side_to_move=row.side_to_move)
    display(HTML(f"<h3>{html.escape(text_label)}</h3><pre style='white-space:pre-wrap'>{html.escape(shown_text)}</pre>"))

    sentence = str(row["sentence"])
    claim = str(row["claim_text"])
    source_quote = str(row.get("source_quote", "")).strip()
    source_subject = str(row.get("source_subject", "")).strip()
    claim_in_sentence = normalize_for_match(claim) in normalize_for_match(sentence)
    claim_in_explanation = normalize_for_match(claim) in normalize_for_match(body)
    if not claim_in_sentence:
        warning = "Step 4's claim_text is not a verbatim substring of the sentence associated with it."
        if source_quote:
            warning += " The exact source quote and any added subject are shown separately below."
        elif not claim_in_explanation:
            warning += " It is also not a verbatim substring of the original explanation."
        display(HTML(f"<p style='color:#9a3412;background:#fff7ed;padding:10px'><b>Extraction warning:</b> {html.escape(warning)} Add a split-mismatch note; use unknown if the intended claim is unclear.</p>"))
    display(HTML(f"<h3>Sentence Step 4 associated with this claim</h3><p>{html.escape(sentence)}</p>"))
    if source_quote:
        display(HTML(f"<h3>Exact source quote selected by the splitter</h3><p>{html.escape(source_quote)}</p>"))
    if source_subject:
        display(HTML(f"<h3>Subject added by the splitter</h3><p>{html.escape(source_subject)}</p>"))
    display(HTML(f"<h3>Claim text to rate</h3><p><b>{html.escape(claim)}</b></p>"))

In [3]:
CONTEXT_SHEET_PATH = STEP4_DIR / "annotation_sheet_with_context.csv"
CONTEXT_SOURCE_COLUMNS = ["blind_id", "fen", "side_to_move", "move_to_explain_san", "full_llm_explanation", "sentence", "source_quote", "source_subject", "claim_text", "claim_text_in_associated_sentence", "claim_text_in_original_llm_output"]

def create_context_sheet():
    rows = []
    for blind_id in sheet.index:
        row = sheet.loc[blind_id]
        hidden_key = crosswalk.loc[blind_id]
        generation_id = (str(hidden_key["model"]), str(hidden_key["condition"]), int(hidden_key["repeat"]), str(hidden_key["pilot_id"]))
        generated = generations[generation_id]
        explanation = explanation_without_self_verdict(str(generated["text"]))
        sentence = str(row["sentence"])
        claim = str(row["claim_text"])
        in_sentence = normalize_for_match(claim) in normalize_for_match(sentence)
        in_original_output = normalize_for_match(claim) in normalize_for_match(str(generated["text"]))
        rows.append({
            "blind_id": blind_id,
            "fen": row["fen"],
            "side_to_move": row["side_to_move"],
            "move_to_explain_san": row["move_to_explain_san"],
            "full_llm_explanation": explanation,
            "sentence": sentence,
            "source_quote": row.get("source_quote", ""),
            "source_subject": row.get("source_subject", ""),
            "claim_text": claim,
            "claim_text_in_associated_sentence": in_sentence,
            "claim_text_in_original_llm_output": in_original_output,
            "verdict": row.get("verdict", ""),
            "category_manual": row.get("category_manual", ""),
            "notes": row.get("notes", ""),
        })

    result = pd.DataFrame(rows)
    candidate = CONTEXT_SHEET_PATH
    suffix = 2
    while candidate.exists():
        existing = pd.read_csv(candidate, dtype={"blind_id": str}, keep_default_na=False)
        if set(CONTEXT_SOURCE_COLUMNS).issubset(existing.columns):
            old_source = existing[CONTEXT_SOURCE_COLUMNS].astype(str).reset_index(drop=True)
            new_source = result[CONTEXT_SOURCE_COLUMNS].astype(str).reset_index(drop=True)
            if old_source.equals(new_source):
                return existing, candidate, False
        candidate = CONTEXT_SHEET_PATH.with_name(f"{CONTEXT_SHEET_PATH.stem}_{suffix}{CONTEXT_SHEET_PATH.suffix}")
        suffix += 1

    result.to_csv(candidate, index=False, encoding="utf-8-sig")
    return result, candidate, True

context_sheet, context_sheet_path, context_sheet_created = create_context_sheet()
mismatch_count = int((~context_sheet["claim_text_in_associated_sentence"]).sum())
action = "Created" if context_sheet_created else "Loaded matching existing"
print(f"{action} {len(context_sheet)} context rows; {mismatch_count} claim(s) are not verbatim in their associated sentence.")
print(f"Context sheet: {context_sheet_path}")
display(context_sheet.head(3))

Created 161 context rows; 0 claim(s) are not verbatim in their associated sentence.
Context sheet: D:\files for Germany\srh\GitHub files\chess-explanations-thesis\outputs\step4_claims\2026-10\annotation_sheet_with_context_5.csv


,blind_id,fen,side_to_move,move_to_explain_san,full_llm_explanation,sentence,source_quote,source_subject,claim_text,claim_text_in_associated_sentence,claim_text_in_original_llm_output,verdict,category_manual,notes
0,C0001,r2q1rk1/ppp1bppp/8/4P2b/1nBPQ3/P1p1BN2/2P3PP/R...,black,Bg6,The human likely moved the bishop from c5 to g...,The human likely moved the bishop from c5 to g...,,,improve the bishop's safety by removing it fro...,True,True,,,
1,C0002,r2B4/p1Pk3p/P5p1/3p4/1bp2P2/3p2P1/1PbN3P/R3K3 ...,white,Rc1,The human likely played Rc1 to connect the roo...,"However, this move doesn't directly challenge ...",,,this move doesn't directly challenge Black's t...,True,True,,,
2,C0003,2r2rk1/1bq2ppp/pp2p3/2n5/1BP1n3/1N2P2P/PQ2BPP1...,black,Rfd8,"The human player chooses Rfd8, moving the rook...",Verdict: good,Verdict: good,,Verdict: good,True,True,,,


In [4]:
# Change this to any blind_id from annotation_sheet_blind.csv, then run this cell.
BLIND_ID = "C0003"
show_claim(BLIND_ID)

In [5]:
# Optional: replace these with any FEN and the move in SAN to inspect another position.
# For example, copy the FEN and move shown above for the currently selected blind claim.
FEN = str(sheet.loc[BLIND_ID, "fen"])
SUGGESTED_MOVE_SAN = str(sheet.loc[BLIND_ID, "move_to_explain_san"])
show_position(FEN, SUGGESTED_MOVE_SAN)

## Save your rating

For each claim, set `BLIND_ID`, then choose `correct`, `incorrect`, or `unknown`. Use `unknown` when the claim cannot be judged from the supplied position and information; it is excluded from the primary error-rate denominator. If Step 5 reports a split mismatch, mention it in `NOTES`; if that leaves the claim ambiguous, use `unknown`. Add an optional manual category and note. Run the next cell once per claim; it updates only that blind ID in the separate Step 5 output file.

In [6]:
ALLOWED_VERDICTS = {"correct", "incorrect", "unknown"}
MANUAL_COLUMNS = ["blind_id", "verdict", "category_manual", "notes"]

def save_manual_annotation(blind_id, verdict, category_manual="", notes=""):
    blind_id = str(blind_id).strip().upper()
    verdict = str(verdict).strip().lower()
    if blind_id not in sheet.index:
        raise KeyError(f"Unknown blind_id {blind_id!r}.")
    if verdict not in ALLOWED_VERDICTS:
        raise ValueError(f"verdict must be one of {sorted(ALLOWED_VERDICTS)}.")

    entries = {}
    if MANUAL_PATH.exists():
        saved = pd.read_csv(MANUAL_PATH, dtype={"blind_id": str}, keep_default_na=False)
        if list(saved.columns) != MANUAL_COLUMNS:
            raise ValueError(f"Unexpected columns in existing annotation file: {list(saved.columns)}")
        if saved["blind_id"].duplicated().any():
            raise ValueError("Existing Step 5 annotation file contains duplicate blind_id values.")
        unknown_ids = set(saved["blind_id"]) - set(sheet.index)
        if unknown_ids:
            raise ValueError(f"Existing Step 5 file has blind IDs absent from the current sheet: {sorted(unknown_ids)}")
        entries = {item["blind_id"]: item for item in saved.to_dict(orient="records")}

    entries[blind_id] = {
        "blind_id": blind_id,
        "verdict": verdict,
        "category_manual": str(category_manual).strip(),
        "notes": str(notes).strip(),
    }
    order = {item: index for index, item in enumerate(sheet.index)}
    result = pd.DataFrame(sorted(entries.values(), key=lambda item: order[item["blind_id"]]), columns=MANUAL_COLUMNS)
    result.to_csv(MANUAL_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved {len(result)} rating(s) to {MANUAL_PATH}")

# Set a verdict below, then run this cell. Leaving it blank makes Run All safe.
VERDICT = ""  # correct | incorrect | unknown
CATEGORY_MANUAL = ""
NOTES = ""
if VERDICT.strip():
    save_manual_annotation(BLIND_ID, VERDICT, CATEGORY_MANUAL, NOTES)
else:
    print("No rating saved yet. Set VERDICT and rerun this cell after reviewing a claim.")

No rating saved yet. Set VERDICT and rerun this cell after reviewing a claim.
